## Install Required Libraries
First, we need to install the libraries used for data loading, audio processing, and model evaluation.

In [ ]:
pip install datasets scikit-learn joblib librosa soundfile

## Import Libraries, Load Model, and Define Pipeline
Next, we'll import the necessary Python libraries and load your pre-trained **classifier** and **scaler** from the `/content/audio_deepfake_model.joblib` file. Based on your saving method, this file is expected to contain a dictionary with the keys `'classifier'` and `'scaler'`.

Since your original model had a `pipeline.process_audio_file` method for feature extraction, we'll define a placeholder `AudioDeepfakePipeline` class that wraps the loaded classifier. **You will need to fill in the implementation for the `process_audio_file` method** with your exact feature extraction logic (Vocal Tract + MFCC + Centroid) that was used during training. This is crucial for the evaluation to proceed correctly.

In [ ]:
import joblib
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report
import librosa
import soundfile as sf
# Assuming RandomForestClassifier and StandardScaler were the types used based on kernel state
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import StandardScaler

# Define a placeholder class for the pipeline that will hold the classifier
# and provide a method for feature extraction.
class AudioDeepfakePipeline:
    def __init__(self, classifier_model):
        self.classifier = classifier_model

    def process_audio_file(self, audio_array):
        """
        Placeholder for the audio feature extraction method.
        This method was part of the original pipeline but was not saved.
        It should extract Vocal Tract + MFCC + Centroid features from audio_array.
        """
        raise NotImplementedError(
            "The `process_audio_file` method for feature extraction is missing. "
            "Please define the logic to extract 'Vocal Tract + MFCC + Centroid' features "
            "from the `audio_array` and return them. "
            "Example: return your_feature_extraction_function(audio_array)"
        )

# Load the trained classifier and scaler
# The joblib file is expected to contain a dictionary with 'classifier' and 'scaler' keys.
tried_loading_model = False
try:
    loaded_model_data = joblib.load('/content/audio_deepfake_model.joblib')
    classifier = loaded_model_data['classifier']  # Correctly load classifier
    scaler = loaded_model_data['scaler']      # Correctly load scaler
    tried_loading_model = True
    print("Classifier and scaler loaded successfully!")

    # Instantiate the custom pipeline with the loaded classifier
    pipeline = AudioDeepfakePipeline(classifier)

except KeyError as e:
    print(f"Error: The joblib file did not contain the expected key: {e}.")
    print("Please ensure your saved model file contains both the trained 'classifier' and 'scaler'.")
    print("The structure should be: `joblib.dump({'classifier': classifier_obj, 'scaler': scaler_obj}, 'audio_deepfake_model.joblib')`")
except FileNotFoundError:
    print("Error: Model file '/content/audio_deepfake_model.joblib' not found.")
    print("Please ensure the model file is correctly uploaded to the specified path.")
except Exception as e:
    print(f"An unexpected error occurred during model loading: {e}")

if not tried_loading_model:
    # Fallback to mock objects if loading failed, to allow subsequent cells to run partially
    print("Using mock objects due to loading failure. Feature extraction will not work correctly.")

    class MockClassifier:
        def predict(self, features):
            print("MockClassifier: Returning dummy predictions.")
            return np.random.randint(0, 2, size=len(features))

    class MockScaler:
        def transform(self, data):
            print("MockScaler: Returning data without scaling.")
            return data

    class MockPipeline:
        def __init__(self):
            self.classifier = MockClassifier()
        def process_audio_file(self, audio_array):
            print("MockPipeline: Returning dummy features as process_audio_file is not implemented.\nPlease implement your feature extraction logic.")
            # Return a 2D array for scaler.transform to avoid shape errors later
            return np.random.rand(1, 10)

    pipeline = MockPipeline()
    scaler = MockScaler()
    classifier = MockClassifier() # Also define classifier for completeness if used directly elsewhere


Classifier and scaler loaded successfully!


In [ ]:
import numpy as np
import scipy.signal as signal
import scipy.linalg
import librosa
from sklearn.ensemble import RandomForestClassifier

class MLVocalTractPipeline:

    def __init__(self, sample_rate=16000, n_tubes=6, lpc_order=12):
        self.sample_rate = sample_rate
        self.n_tubes = n_tubes
        self.lpc_order = lpc_order
        self.classifier = RandomForestClassifier(
            n_estimators=1500, random_state=42, max_features='sqrt', n_jobs=1
        )
        self.feature_dim = self.n_tubes + (self.n_tubes - 1) + 1

    def extract_features_from_frame(self, frame):
        frame_filt = frame[1:] - 0.97 * frame[:-1]
        frame_windowed = frame_filt * np.hamming(len(frame_filt))

        autocorr = signal.correlate(frame_windowed, frame_windowed, mode="full")
        autocorr = autocorr[
            len(frame_windowed) - 1 : len(frame_windowed) - 1 + self.lpc_order + 1
        ]

        if np.max(autocorr) < 1e-6:
            return np.zeros(self.feature_dim)

        try:
            a = scipy.linalg.solve_toeplitz(
                (autocorr[:-1], autocorr[:-1]), autocorr[1:]
            )
            k = -a
        except scipy.linalg.LinAlgError:
            return np.zeros(self.feature_dim)

        areas = np.ones(self.n_tubes)
        for i in range(min(len(k), self.n_tubes - 1)):
            k_val = np.clip(k[i], -0.95, 0.95)
            areas[i + 1] = areas[i] * ((1.0 + k_val) / (1.0 - k_val))

        area_diffs = np.diff(areas)
        max_ratio = np.max(areas) / (np.min(areas) + 1e-6)

        return np.hstack([areas, area_diffs, [max_ratio]])

    def process_audio_file(self, audio_array, frame_duration=0.03):
        audio_trimmed, _ = librosa.effects.trim(audio_array, top_db=20)
        if len(audio_trimmed) < int(self.sample_rate * frame_duration):
            audio_trimmed = audio_array

        frame_length = int(self.sample_rate * frame_duration)
        n_frames = len(audio_trimmed) // frame_length

        total_dim = self.feature_dim + (13 * 3) + 2
        if n_frames == 0:
            return np.zeros(total_dim)

        vocal_features = []
        for i in range(n_frames):
            frame = audio_trimmed[i * frame_length : (i + 1) * frame_length]
            feat = self.extract_features_from_frame(frame)
            vocal_features.append(feat)
        vocal_mean = np.mean(vocal_features, axis=0)

        mfcc = librosa.feature.mfcc(
            y=audio_trimmed, sr=self.sample_rate, n_mfcc=13
        )
        mfcc_d = librosa.feature.delta(mfcc)
        mfcc_dd = librosa.feature.delta(mfcc, order=2)

        mfcc_stacked = np.hstack(
            [
                np.mean(mfcc, axis=1),
                np.mean(mfcc_d, axis=1),
                np.mean(mfcc_dd, axis=1),
            ]
        )

        spec_centroid = np.mean(
            librosa.feature.spectral_centroid(
                y=audio_trimmed, sr=self.sample_rate
            )
        )
        log_energy = np.log(np.sum(audio_trimmed**2) + 1e-6)

        return np.hstack([vocal_mean, mfcc_stacked, [spec_centroid, log_energy]])

pipeline = MLVocalTractPipeline()

## Define Audio Preprocessing Function
This function will resample the raw audio to 16kHz and apply peak normalization, as described in your provided evaluation snippet.

In [ ]:
def extract_numpy_audio(raw_audio, target_sample_rate=16000):
    # raw_audio is expected to be a dictionary from the Hugging Face datasets audio feature,
    # containing 'array' (audio data) and 'sampling_rate'.
    audio_data = raw_audio['array']
    original_sr = raw_audio['sampling_rate']

    # Resample if necessary
    if original_sr != target_sample_rate:
        audio_data = librosa.resample(y=audio_data, orig_sr=original_sr, target_sr=target_sample_rate)

    # Peak Normalize (to avoid clipping and standardize amplitude)
    max_abs_val = np.max(np.abs(audio_data))
    if max_abs_val > 0:
        audio_data = audio_data / max_abs_val

    return audio_data

## Load and Evaluate Unseen Data
Now, we will load the specified dataset, preprocess the audio samples, extract features, apply the scaler, make predictions using the loaded model, and finally print the accuracy score and classification report.

In [ ]:
print("Loading Unseen Evaluation Data from Dataset B (Indices 0..300)...")
ds_b_eval = (
    load_dataset("3004lakshu/Deepfake-Audio", split="train", streaming=True)
    .shuffle(seed=42, buffer_size=1000) # Buffer size limits memory usage during shuffle
    .take(300)                          # Efficiently streams only the first 300 files
)

X_eval, y_eval = [], []

for i, item in enumerate(ds_b_eval):
    raw_audio = item["audio"]
    # 1. Resample to 16kHz + Peak Normalize
    audio_array = extract_numpy_audio(raw_audio)

    # 2. Extract Vocal Tract + MFCC + Centroid features
    # Assuming pipeline has a method 'process_audio_file'
    feat = pipeline.process_audio_file(audio_array)

    # 3. Align Label (1 = Real -> 0, 0 = Fake -> 1)
    raw_label = item["label"]
    label = 0 if int(raw_label) == 1 else 1

    X_eval.append(feat)
    y_eval.append(label)

    if (i + 1) % 10 == 0:
        print(f"Processed {i+1} samples...") # Removed len(ds_b_eval) from here

X_eval, y_eval = np.array(X_eval), np.array(y_eval)

# Ensure X_eval has the correct shape for scaling
# If features are 1D for each audio, reshape to 2D for scaler.transform
if X_eval.ndim == 1:
    X_eval = np.array(list(X_eval)).reshape(len(X_eval), -1)


# 4. Transform features using existing scaler
X_eval_scaled = scaler.transform(X_eval)

# 5. Predict using trained classifier
eval_predictions = pipeline.classifier.predict(X_eval_scaled)

# 6. Evaluation Output
print("\n--- Evaluation on Strictly Unseen Dataset B Samples ---")
print("Accuracy Score:", accuracy_score(y_eval, eval_predictions))
print(
    "\nClassification Report:\n",
    classification_report(y_eval, eval_predictions),
)


Loading Unseen Evaluation Data from Dataset B (Indices 0..300)...


Resolving data files:   0%|          | 0/604 [00:00<?, ?it/s]

Processed 10 samples...
Processed 20 samples...
Processed 30 samples...
Processed 40 samples...
Processed 50 samples...
Processed 60 samples...
Processed 70 samples...
Processed 80 samples...
Processed 90 samples...
Processed 100 samples...
Processed 110 samples...
Processed 120 samples...
Processed 130 samples...
Processed 140 samples...
Processed 150 samples...
Processed 160 samples...
Processed 170 samples...
Processed 180 samples...
Processed 190 samples...
Processed 200 samples...
Processed 210 samples...
Processed 220 samples...
Processed 230 samples...
Processed 240 samples...
Processed 250 samples...
Processed 260 samples...
Processed 270 samples...
Processed 280 samples...
Processed 290 samples...
Processed 300 samples...


NotFittedError: This RandomForestClassifier instance is not fitted yet. Call 'fit' with appropriate arguments before using this estimator.